# EOR GRF Sky Model Generation: Physics & Mathematics

**Comprehensive documentation of the physics and mathematics behind the EOR (Epoch of Reionization) Gaussian Random Field (GRF) sky model generation pipeline.**

This notebook explains:
1. The cosmological power spectrum model
2. How frequency-frequency covariance is computed
3. Gaussian random realization generation
4. Conversion to HEALPix sky maps and .skyh5 format
5. Brightness temperature scaling and units

## 1. Power Spectrum Model

### 1.1 Parameterized Gaussian Power Spectrum

The EOR signal is modeled as a 3D Gaussian random field with a power law spectrum. The code uses the `redshifted_gaussian_fields` package's `ParameterizedGaussianPowerSpectrum` class:

$$P(k) = \sum_{i} a_i \cdot G(k; k_{0,i}, \sigma_i)$$

where:
- **$a_i$**: Amplitude coefficients for each Gaussian component
- **$k_{0,i}$**: Central wavenumber (in $h$ Mpc$^{-1}$)
- **$\sigma_i$**: Width of the Gaussian
- **Gaussian term**: $G(k; k_0, \sigma) = \exp\left(-\frac{(k-k_0)^2}{2\sigma^2}\right)$

### 1.2 H6C Configuration (from grf_covariance.py)

```python
k0 = np.logspace(-2., 1., 11)  # [0.01, 0.13, 0.18, ..., 10] h/Mpc
a = k0**(-2.7)                 # Power law: a ∝ k^-2.7

normalization_point = 0.2      # k_norm
normalization_amplitude = 1.   # P_norm
term_type = 'flat_gauss'       # Uses flat Gaussian form
```

### 1.3 Physical Interpretation

- **Power law index -2.7**: Steeper than white noise (-2.0). Represents clustering on small scales (relevant to EOR ionization bubbles)
- **Multiple Gaussian peaks**: Captures structure across different k-scales from large (k~0.01) to small (k~10) scales
- **Normalization**: Set at $k=0.2$ h/Mpc, $P(0.2)=1$ Jy²/sr² (approximately)
- **Scales covered**:
  - Large scale: k = 0.01 h/Mpc → ~6 Mpc (comoving, z~6)
  - Small scale: k = 10 h/Mpc → ~0.06 Mpc (comoving, z~6)

## 2. Frequency-Frequency Covariance Matrix

### 2.1 The Covariance Generation Problem

The EOR signal has correlations across frequencies (redshifts). These are captured in the **cross-frequency angular power spectrum covariance matrix**.

Given frequencies $\nu_1, \nu_2$ and angular scale $\ell$:

$$C_{\ell}(\nu_1, \nu_2) = \int_0^\infty dk\, W(k;\nu_1,\nu_2) \times P_3D(k)$$

where:
- **$P_{3D}(k)$**: The 3D power spectrum (from section 1)
- **$W(k;\nu_1,\nu_2)$**: Window function relating 3D wavenumber $k$ to 2D angular scale $\ell$ and frequency separation
- **$C_\ell(\nu_1,\nu_2)$**: Angular power spectrum in Jy²/sr² (for intensity maps)

### 2.2 Cosmological Integration

The covariance computation integrates over 3D k-space using cosmological transformations:

$$k_\perp = \frac{\ell}{D_C(z)}$$
$$k_\parallel = \frac{2\pi \Delta\nu}{\partial D_C / \partial \nu}$$

where:
- **$D_C(z)$**: Comoving distance at redshift $z$
- **$\Delta\nu$**: Frequency separation (bandwidth element)
- **Planck15 cosmology**: Used internally (`astropy.cosmology.Planck15`)
  - $H_0 = 67.74$ km/s/Mpc
  - $\Omega_m = 0.3075$
  - $\Omega_\Lambda = 0.6925$

### 2.3 H6C Pipeline (grf_covariance.py)

**Input parameters:**
- **Frequency axis**: $\nu \in [76, 229]$ MHz (H6C band) in $\Delta\nu = 76.3$ kHz steps
- **Angular scale ($\ell$)**: $\ell \in [0, \ell_{\max}]$ where $\ell_{\max} = 1250$ (default)
- **Precision**: $N_p = 15$ Legendre polynomial terms, $\epsilon = 10^{-15}$ tolerance

**Output:**
- Covariance matrix shape: `(n_freqs, n_freqs, n_ell)`
- Dimensions: ~2008 frequencies × 2008 frequencies × 1251 $\ell$ modes
- Stored in `covariance.h5` using HDF5 format
- File size: ~100-500 MB depending on $\ell_{\max}$

### 2.4 Covariance Structure

Key properties:
- **Diagonal correlations**: $C_\ell(\nu, \nu)$ peaks near $\ell$-dependent scales
- **Off-diagonal**: $C_\ell(\nu_1, \nu_2)$ decays for $|\nu_1 - \nu_2| \gg$ coherence width
- **Coherence frequency**: ~1-10 MHz (for EOR at z~6-8), depends on $\ell$
- **Positive semi-definite**: Ensures valid covariance matrix for Gaussian realizations

## 3. Gaussian Random Field Realization

### 3.1 Mathematical Foundation

Given a covariance matrix $\mathbf{C} \in \mathbb{R}^{N_f \times N_f}$, generate correlated Gaussian samples:

$$\mathbf{x} = \mathbf{L} \mathbf{z}$$

where:
- **$\mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$**: Standard Gaussian random vector (from numpy/random with seed)
- **$\mathbf{L}$**: Cholesky decomposition of covariance: $\mathbf{C} = \mathbf{L} \mathbf{L}^T$
- **$\mathbf{x}$**: Correlated Gaussian field with covariance $\mathbb{E}[\mathbf{x}\mathbf{x}^T] = \mathbf{C}$

### 3.2 GRF Properties

- **Mean**: 0 (by definition)
- **Variance**: $\sigma^2 = C(\nu, \nu)$, varies with frequency
- **Spatial statistics**: Gaussian with power spectrum $P(k)$ from section 1
- **Reproducibility**: Controlled by random seed (default: 2038)

### 3.3 H6C Realization Pipeline (grf_realization.py)

```bash
rgf realization --nside 256 --seed 777 --low-memory \
  --covpath sky_models/raw/covariance.h5 \
  --outpath sky_models/raw/eor-grf-nside256.h5 --overwrite
```

**Process:**
1. Load covariance matrix from `covariance.h5`
2. Perform Cholesky decomposition: $\mathbf{C} = \mathbf{L} \mathbf{L}^T$
3. Generate standard Gaussian: $\mathbf{z}_\ell \sim \mathcal{N}(0,1)$ for each $\ell$ mode
4. Create correlated field: $\mathbf{x}_\ell = \mathbf{L}_\ell \mathbf{z}_\ell$
5. Inverse spherical harmonic transform: Convert $\{x_\ell\}_{\ell=0}^{\ell_{max}}$ to spatial sky map
6. Inverse Fourier transform in frequency: Get frequency-dependent HEALPix maps

**Output:**
- HDF5 file: `eor-grf-nside256.h5`
- Dataset: `healpix_maps` with shape `(n_freqs, n_pixels)`
- Values: Brightness temperature in Jy/sr
- Dimensions: ~2008 frequencies × 196,608 HEALPix pixels (nside=256)

**SLURM Configuration:**
- 1 node, 16 tasks, 48 GB RAM, 4-hour walltime
- `--low-memory` flag: Uses disk caching for large intermediate arrays
- Required for nside ≥ 256

## 4. HEALPix Sky Map Format & Units

### 4.1 HEALPix Pixelization

The sphere is divided into $N_{pix} = 12 \times N_{side}^2$ equal-area pixels:

$$N_{pix}(N_{side}) = 12 N_{side}^2$$

| nside | $N_{pix}$ | $\Delta\Omega$ (arcmin²) | Use Case |
|-------|-----------|------------------------|----------|
| 64    | 49,152    | ~13.5                  | Testing  |
| 128   | 196,608   | ~3.4                   | Fast sims |
| 256   | 786,432   | ~0.85                  | H6C default |
| 512   | 3,145,728 | ~0.21                  | High resolution |

**H6C Choice (nside=256):**
- Pixel scale: ~0.85 arcmin² (30 arcsec)
- Matches HERA beam resolution (~5° at 120 MHz)
- Avoids excessive CPU for small-scale structure

### 4.2 Brightness Temperature

The EOR signal is the **21 cm line emission** from neutral hydrogen:

**Brightness Temperature Definition:**
$$T_b[K] = \frac{\lambda^2}{2 k_B} I_\nu[\text{Jy/sr}]$$

where:
- **$\lambda = c/\nu$**: Wavelength
- **$k_B = 1.38 \times 10^{-23}$ J/K**: Boltzmann constant
- **$I_\nu$**: Specific intensity (Jy/sr) = $10^{-26}$ W m$^{-2}$ Hz$^{-1}$ sr$^{-1}$

**Numerical conversion:**
At 120 MHz (z~8, EOR peak):
$$1 \text{ Jy/sr} = 5.7 \times 10^4 \text{ K}$$

**EOR Brightness Temperature Scales:**
- **Ionized regions**: $T_b \sim -100$ to 0 K (absorption if foreground source is behind)
- **Neutral regions**: $T_b \sim 0$ to 100+ K (emission)
- **RMS fluctuations**: $\Delta T_b \sim 10-50$ K (varies with redshift)

### 4.3 Units in Pipeline

**Throughout the pipeline:**
1. **Covariance computation**: Jy²/sr² (intensity space)
2. **GRF realization**: Jy/sr (intensity map)
3. **HEALPix output** (`healpix_maps` in .h5): Jy/sr
4. **Sky model** (.skyh5): Jy/sr (stokes I component)
5. **Visibility simulation**: Jy/sr → Jy (via antenna beam integral)

**No automatic brightness temperature conversion** in the pipeline. Conversion to K requires frequency information and is done post-simulation if needed.

## 5. Sky Model Generation (.skyh5)

### 5.1 PyRadioSky SkyModel Format

The `.skyh5` format (from `pyradiosky` package) stores:

```yaml
SkyModel:
  component_type: "healpix"  # HEALPix diffuse component
  nside: 256                 # HEALPix resolution
  hpx_order: "ring"         # Ring ordering (vs. nested)
  hpx_inds: [0, 1, ..., 786431]  # Pixel indices
  
  stokes: [I, Q, U, V]       # Stokes parameters
                             # Shape: (4, n_freq, n_pix)
  
  freq_array: [76.36 MHz, 76.38 MHz, ...]  # Frequency array
  
  spectral_type: "full"     # Frequency-dependent (not power law)
  frame: "icrs"             # Equatorial coordinates
```

### 5.2 Per-Channel Generation (from sky_model.py)

```python
def make_grf_eor_model(model_file, channels, label=""):
    # Load covariance/realization HDF5
    with h5py.File(model_dir / model_file, 'r') as fl:
        healpix_maps = fl['healpix_maps'][fch]  # (n_pix,) array
    
    # Apply offset for simulator compatibility
    if offset_mode == "shift_min":
        min_val = healpix_maps.min()
        if min_val < floor_epsilon:
            shift = floor_epsilon - min_val
            healpix_maps += shift  # Ensure T_b >= 1e-6 Jy/sr
    
    # Create SkyModel with units
    eor_model.stokes[0, 0] = healpix_maps * units.Jy / units.sr
    eor_model.freq_array[0] = freqs[fch]
    
    # Write per-frequency .skyh5
    eor_model.write_skyh5(f"sky_models/eor-grf-256/fch{fch:04d}.skyh5")
```

### 5.3 Offset Strategy

**Problem**: Gaussian realization has negative values (centered at 0 mean). Matvis simulator may not handle negative brightness temperatures properly.

**Solutions implemented:**

| Mode | Effect | Equation |
|------|--------|----------|
| `none` | No modification | $T_b^{out} = T_b^{raw}$ |
| `constant` | Add fixed offset | $T_b^{out} = T_b^{raw} + \text{offset\_value}$ |
| `shift_min` | Shift to floor | $T_b^{out} = T_b^{raw} + \max(0, \epsilon - \min(T_b))$ |

**H6C Default**: `shift_min` with $\epsilon = 10^{-6}$ Jy/sr
- Preserves relative amplitude variations
- Avoids brightness inversion
- Keeps all pixels ≥ 1 μJy/sr

### 5.4 Output Structure

**Directory layout:**
```
sky_models/eor-grf-256/
├── fch0227.skyh5  (76.36 MHz, z ≈ 6.5)
├── fch0228.skyh5  (76.38 MHz)
├── ...
└── fch0316.skyh5  (228.5 MHz, z ≈ 1.3)
```

**Per-file size**: ~5-10 MB (depending on nside)
**Total for all H6C channels** (~90): ~500 MB - 1 GB

## 6. Complete Pipeline Summary

### 6.1 Three-Stage Generation

```
┌─────────────────────────────────────────────────────────────┐
│ Stage 1: GRF Covariance Generation (grf-covariance)          │
│                                                              │
│  Input: Power spectrum params (a, k₀, normalization)       │
│         Frequency range, ℓ_max                             │
│  ↓                                                          │
│  • Setup GaussianCosmologicalFieldGenerator                 │
│  • Integrate P_3D(k) over k-space → C_ℓ(ν₁,ν₂)            │
│  • Output: covariance.h5 (100-500 MB)                      │
│  ↓                                                          │
│  Runtime: ~30-60 minutes on NRAO hera partition            │
└─────────────────────────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────┐
│ Stage 2: GRF Realization (grf-realization)                   │
│                                                              │
│  Input: covariance.h5, nside, seed                          │
│  ↓                                                          │
│  • Load frequency-frequency covariance matrix               │
│  • For each ℓ mode:                                          │
│    - Sample z ~ N(0,1)                                      │
│    - Solve x = L·z (Cholesky decomposition)                 │
│    - iFFT → spatial HEALPix map                            │
│  • Output: eor-grf-nside256.h5 (shape: freq × 786k pixels) │
│  ↓                                                          │
│  Runtime: 2-4 hours on NRAO hera partition (nside=256)      │
└─────────────────────────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────┐
│ Stage 3: Sky Model Generation (sky-model grf-eor)            │
│                                                              │
│  Input: eor-grf-nside256.h5, channel list                   │
│  ↓                                                          │
│  For each channel:                                          │
│  • Load healpix_maps[fch] from HDF5                         │
│  • Apply offset (shift_min with ε=1e-6 Jy/sr)             │
│  • Create pyradiosky.SkyModel (HEALPix format)              │
│  • Write .skyh5 file (per frequency)                        │
│  ↓                                                          │
│  Output: sky_models/eor-grf-256/fch*.skyh5 (~500 MB total) │
│  ↓                                                          │
│  Runtime: 5-10 minutes (single node)                        │
└─────────────────────────────────────────────────────────────┘
```

### 6.2 Key Physics Insights

| Aspect | Value/Behavior | Physical Reason |
|--------|-----------------|------------------|
| **Power law index** | -2.7 | EOR ionization structure |
| **Scales covered** | 0.01-10 h/Mpc | Bubble scales (1-600 Mpc comoving) |
| **Frequency coherence** | ~1-10 MHz | Redshift dimension of bubbles |
| **Brightness scale** | 10-100 K (Jy/sr) | 21 cm line optical depth |
| **Negative T_b** | Common | Absorption by foreground sources |
| **Offset shifting** | shift_min | Simulator numerical stability |

### 6.3 Mathematical Chain

$$\text{Power spectrum} \xrightarrow{\text{Cosmology}} \text{Covariance matrix} \xrightarrow{\text{Cholesky}} \text{Gaussian field} \xrightarrow{\text{FFT}} \text{Sky map} \xrightarrow{\text{.skyh5}} \text{Simulator}$$

### 6.4 Common Configurations

**Default H6C (Recommended):**
```bash
./vsim.py grf-covariance --ell-max 1250
./vsim.py grf-realization --nside 256 --seed 777
./vsim.py sky-model grf-eor --nside 256 --channels 227~316
```

**Low Memory (Test/Development):**
```bash
./vsim.py grf-covariance --ell-max 500 --local --test-mode
./vsim.py grf-realization --nside 128 --seed 777 --local
```

**High Resolution (Research):**
```bash
./vsim.py grf-realization --nside 512 --seed 777
```

## 7. Technical Implementation Details

### 7.1 Code Flow (vsim.py → Python → CLI)

**Command 1: grf-covariance**
```
vsim.py
  ↓
@slurmify decorator (slurm.py)
  ↓
run_compute_grf_covariance() [grf_covariance.py]
  ├─ Create GaussianCosmologicalFieldGenerator
  ├─ Call gcfg.compute_cross_frequency_angular_power_spectrum()
  └─ Save to covariance.h5
  ↓
Subprocess: "rgf covariance [args]" (external CLI from redshifted_gaussian_fields)
```

**Command 2: grf-realization**
```
vsim.py
  ↓
@slurmify decorator
  ↓
run_compute_grf_realization() [grf_realization.py]
  ↓
Subprocess: "rgf realization [args]" (external CLI)
  ├─ Load covariance.h5
  ├─ For each ℓ: solve Cholesky → Gaussian sample
  ├─ Inverse transforms
  └─ Write eor-grf-nside*.h5
```

**Command 3: sky-model grf-eor**
```
vsim.py
  ↓
make_grf_eor_model() [sky_model.py]
  ├─ Load healpix_maps from HDF5
  ├─ Apply offset transformation
  ├─ Create pyradiosky.SkyModel
  └─ Write per-frequency .skyh5 files
```

### 7.2 File Dependencies

```
covariance.h5
  ↓ (input to Stage 2)
eor-grf-nside256.h5
  ↓ (input to Stage 3)
sky_models/eor-grf-256/
  ├── fch0227.skyh5
  ├── fch0228.skyh5
  └── ...
  ↓ (input to Stage 4: visibility simulation)
outputs/[layout]/[beam_config]/...
  └── *.uvh5 (visibility data)
```

### 7.3 Numerical Precision

**Covariance computation:**
- Legendre polynomial order: $N_p = 15$
- Integration tolerance: $\epsilon = 10^{-15}$
- Ensures sub-Jy² precision

**Cholesky decomposition:**
- Uses standard NumPy linear algebra
- Condition number checked (warnings if ill-conditioned)
- 64-bit float (double precision)

**HEALPix pixelization:**
- Ring ordering (sequential frequency access)
- 32-bit or 64-bit float output (configurable)


## 8. Physics Validation & Diagnostics

### 8.1 Covariance Matrix Properties

**Check 1: Symmetry**
$$C(\nu_1, \nu_2, \ell) = C(\nu_2, \nu_1, \ell)$$
Should be exact (within numerical precision)

**Check 2: Positive Semidefinite**
- All eigenvalues $\lambda_i \geq 0$
- Critical for valid Gaussian realization

**Check 3: Diagonal Dominance**
$|C(\nu, \nu, \ell)| > |C(\nu_1, \nu_2, \ell)|$ 
for 
$\nu_1 \neq \nu_2$
- Ensures covariance structure

### 8.2 GRF Realization Diagnostics

**RMS Brightness Temperature per Frequency**
$$\sigma(\nu) = \sqrt{\langle T_b^2(\nu) \rangle}$$
- Should match covariance diagonal: $\sigma^2(\nu) = C(\nu, \nu, \ell=0)$
- Diagnostic: compare input covariance vs. realization RMS

**Power Spectrum Recovery**
$$P_{measured}(k) \approx P_{input}(k)$$
- Compute from realization using power spectrum analysis (hera_pspec)
- Should match input power law within cosmic variance

**Gaussian Statistics Test**
- Pixel value distribution should be Gaussian (with offset shift)
- QQ-plot or Anderson-Darling test

### 8.3 Sky Model Diagnostics

**Check 1: No NaN/Inf**
```python
assert np.all(np.isfinite(skymodel.stokes[0]))
```

**Check 2: Offset Applied Correctly**
```python
assert np.min(skymodel.stokes[0]) >= 1e-6  # Jy/sr floor
```

**Check 3: Frequency Continuity**
- No sudden jumps between adjacent frequency channels
- Implies smooth underlying EOR signal

### 8.4 Output Inspection Script

```python
import h5py
import numpy as np

# Covariance statistics
with h5py.File('sky_models/raw/covariance.h5', 'r') as f:
    cov = f['covariance_matrix'][:]
    print(f"Covariance shape: {cov.shape}")
    print(f"Min: {np.min(cov)}, Max: {np.max(cov)}")
    print(f"Diagonal mean: {np.mean(np.diag(cov))}")

# GRF realization statistics
with h5py.File('sky_models/raw/eor-grf-nside256.h5', 'r') as f:
    maps = f['healpix_maps'][:]
    print(f"Maps shape: {maps.shape}")
    print(f"RMS per frequency (first 10): {np.std(maps[:10], axis=1)}")
    print(f"Min/Max range: {np.min(maps)} to {np.max(maps)}")

# Sky model check
from pyradiosky import SkyModel
sky = SkyModel.from_skyh5('sky_models/eor-grf-256/fch0250.skyh5')
print(f"Stokes I min/max: {np.min(sky.stokes[0])} to {np.max(sky.stokes[0])}")
```

## 9. Key References & Citations

### 9.1 Theory References

1. **Gaussian Random Field Theory**
   - Bardeen et al. (1986) - GRF statistics in cosmology
   - Dodelson (2003) - Modern Cosmology (Chapter on large-scale structure)

2. **EOR 21 cm Radiation**
   - Furlanetto et al. (2006) - Cosmology of the Cosmic Dawn
   - Pritchard & Loeb (2012) - 21 cm as a probe of the first galaxies

3. **Redshifted Gaussian Fields Package**
   - https://github.com/steven-murray/redshifted_gaussian_fields
   - Murray et al. implementation of cosmological field generation

### 9.2 Code Implementation References

- **validation-sim repository**: `/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/`
- **Power spectrum parameters** (grf_covariance.py):
  ```
  k0 = np.logspace(-2., 1., 11)  # [0.01 ... 10] h/Mpc
  a = k0**(-2.7)                 # Power law amplitude
  normalization: (0.2, 1.0)      # k_norm, P_norm
  ```

- **Cosmology** (Planck 2015):
  ```python
  from astropy.cosmology import Planck15
  H0=67.74, Om_m=0.3075, Om_L=0.6925
  ```

### 9.3 Related Notebooks

- `eor_grf_workflow_detailed.ipynb`: Full pipeline walkthrough with code
- `analytic_beam_workflow.ipynb`: Beam modeling for visibility simulation
- `output_vis_waterfall_check.ipynb`: Visibility diagnostic plots

## Summary

The EOR GRF sky model generation pipeline converts theoretical cosmology into realistic, frequency-dependent synthetic observations:

1. **Physics**: Power spectrum model with -2.7 power law captures EOR ionization structure across scales 1-600 Mpc (comoving)

2. **Mathematics**: Frequency-frequency covariance matrix encodes 21 cm radiation correlations across redshifts via cosmological integral transforms

3. **Implementation**: Three-stage pipeline (covariance → realization → sky maps) using external `redshifted_gaussian_fields` package with validation-sim CLI integration

4. **Output**: Per-frequency HEALPix maps (nside=256, ~786k pixels) in standard .skyh5 format, ready for visibility simulation with HERA array

5. **Validation**: Gaussian statistics preserved, brightness range handled via offset shifting, frequency coherence maintained by covariance structure